In [11]:
%pip install -U langchain_groq

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 137 kB 1.5 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from langchain_community.document_loaders import PyPDFLoader

/Users/sawals/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/sawals/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [31]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate



In [13]:
model = ChatGroq(model="openai/gpt-oss-20b",streaming="True")

In [14]:
pdfloader = PyPDFLoader("../data/rag-pdf.pdf")

pdfs = pdfloader.load()

In [16]:
splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=50)
splitted_data=splitter.split_documents(pdfs)

In [17]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [18]:
vector_store= Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings,
    persist_directory="./vector_db"
)

In [24]:
query="Solar System"
data=vector_store.similarity_search(query=query)


In [25]:
context=""
for doc in data:
    context+=doc.page_content +"\n"

print(context)

Question: What is the Solar System?
Answer: It is the Sun and the planets, moons, dwarf planets, asteroids, comets,
and other objects held in orbit by gravity.
The Solar System
Reference document for basic RAG practice
Overview
The Solar System consists of the Sun and all the objects that travel around it.
These objects include eight planets, their moons, dwarf planets, asteroids,
comets, and smaller pieces of rock and ice.
The Sun is a star at the center of the Solar System. Its gravity keeps the
 Pluto is classified as a dwarf planet.
 Earth's rotation causes day and night.
 Earth's axial tilt is the main reason for the seasons.
Common Questions
Question: What is the Solar System?



In [26]:
llm = ChatGroq(model="openai/gpt-oss-20b",streaming="True")

In [ ]:
#response = llm.invoke(
    f"""Can you provide me the answer based on the provided context for my question?
    Context: {context} and Question: {query}
    """
)

#print(response.content)

**Answer:**  
It is the Sun and the planets, moons, dwarf planets, asteroids, comets, and other objects held in orbit by gravity.


In [42]:
def get_context(query: str):
  data = vector_store.similarity_search(query=query)
  context = ""
  for doc in data:
    context += doc.page_content + "\n"

  return {"context": context, "query": query}

In [43]:
prompt = PromptTemplate.from_template(
    """You are a helpful assistant and provide answers based on the context for user question. 
        and if you don't know the answer, then you can say that 'I don't know'. 
        Context: {context} and Question: {query}"""
)

In [44]:
rag_chain= get_context | prompt | llm

In [47]:
response=rag_chain.invoke("How many planets in Solar system?")
print(response.content)

There are **eight** planets in the Solar System.
